In [2]:
import pandas as pd
import statsmodels.formula.api as smf
import scipy.stats as st
import matplotlib.pyplot as plt
from statsmodels.iolib.summary2 import summary_col as sc
import numpy as np

In [3]:
df = pd.read_parquet('/Users/tito/Downloads/data2.parquet')
df.head()

,SERIAL,HHWT,CLUSTER,STRATA,PERNUM,PERWT,NCHILD,SEX,AGE,MARST,...,DC,BLACK,AIAN,API,OTHER,MULTR,HISP,WELFARE,PRSNINC,YRSEDUC
0,4,4.0,2024000000041,280201,1,4.0,0,2,22,6,...,0,0,0,1,0,0,0,0.0,1000.0,13
1,9,74.0,2024000000091,280301,1,74.0,0,2,18,6,...,0,1,0,0,0,0,0,0.0,5100.0,12
2,12,55.0,2024000000121,160001,1,55.0,0,2,19,6,...,0,0,0,0,0,0,0,0.0,0.0,12
3,15,17.0,2024000000151,120201,1,17.0,0,2,19,6,...,0,0,0,0,0,0,0,0.0,6000.0,12
4,19,71.0,2024000000191,10001,1,71.0,0,2,23,6,...,0,0,0,0,0,0,0,0.0,0.0,13


In [4]:
df.columns
df['INCTOT'].min()
df['HINSCAID'].describe()
df['WELFARE'].describe()
df['INSURANCE'] = df['HCOVANY'] - 1 

In [5]:
df['AGE2'] = df['AGE'] ** 2
df['YRSEDUC2'] = df['YRSEDUC'] ** 2
df['INC2'] = df['PRSNINC'] ** 2
df['FINC2'] = df['FTOTINC'] ** 2
df['MEDICAID'] = df['HINSCAID'] - 1 

df['MEDICAGE'] = df['MEDICAID'] * df['AGE']
df['MEDICAGE2'] = df['MEDICAGE'] ** 2

df['INSAGE'] = df['INSURANCE'] * df['AGE']
df['INSAGE2'] = df['INSAGE'] ** 2

In [6]:
model = smf.ols(
    'NCHILD ~ AGE + AGE2 + YRSEDUC + YRSEDUC2 + PRSNINC + INC2 + FTOTINC + FINC2 + INSURANCE + INSAGE + INSAGE2 + MEDICAID + MEDICAGE + MEDICAGE2 + BLACK + HISP + WELFARE + ALABAMA + ALASKA + ARIZONA + ARKANSAS + CALIFORNIA + COLORADO + CONNETICUT + DELAWARE + GEORGIA + HAWAII + IDAHO + ILLINOIS + INDIANA + IOWA + KANSAS + KENTUCKY + LOUISIANA + MAINE + MARYLAND + MASSACHUSETTS + MICHIGAN + MINNESOTA + MISSISSIPPI + MISSOURI + MONTANA + NEBRASKA + NEVADA + NEWHAMPSHIRE + NEWJERSEY + NEWMEXICO + NEWYORK + NORTHCAROLINA + NORTHDAKOTA + OHIO + OKLAHOMA + OREGON + PENNSYLVANIA + RHODEISLAND + SOUTHCAROLINA + SOUTHDAKOTA + TENNESSEE + TEXAS + UTAH + VERMONT + VIRGINIA + WASHINGTON + WESTVIRGINIA + WISCONSIN + WYOMING + DC',
    data=df).fit()

print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                 NCHILD   R-squared:                       0.265
Model:                            OLS   Adj. R-squared:                  0.265
Method:                 Least Squares   F-statistic:                     2827.
Date:                Tue, 10 Nov 2026   Prob (F-statistic):               0.00
Time:                        15:07:36   Log-Likelihood:            -7.8044e+05
No. Observations:              525055   AIC:                         1.561e+06
Df Residuals:                  524987   BIC:                         1.562e+06
Df Model:                          67                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        -4.0566      0.082    -49.482

In [20]:
b = model.params 
se = model.bse

ting = {'BETA':b, 'SE':se}
thang = pd.DataFrame(ting)
thang['T'] = thang['BETA'] / thang['SE']

t = st.t.ppf(q=1-(0.05/2),df=525054)
thang['SIG'] = [0 if -t < i < t else 1 for i in thang['T']]
thang['SIG'].describe()

count    68.000000
mean      0.867647
std       0.341394
min       0.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       1.000000
Name: SIG, dtype: float64

In [24]:
print(thang[thang['SIG'] == 1]['SIG'].count())
print(thang[thang['SIG'] == 0]['SIG'].count())

59
9


BETA   -0.026395
SE      0.023690
T      -1.114161
SIG     0.000000
Name: HAWAII, dtype: float64